# OsteoInsight AI — Colab workflow
This notebook audits the dataset before choosing a workflow. **Do not enable trusted labels until their source and meaning have been verified.** No supervised metric is valid for invented or organizational labels.

In [ ]:
!pip -q install 'torch>=2.2,<3' 'torchvision>=0.17,<1' 'scikit-learn>=1.4,<2' 'pandas>=2.1,<3' 'numpy>=1.26,<3' 'Pillow>=10.2,<13' 'matplotlib>=3.8,<4' 'seaborn>=0.13,<1' 'gradio>=5,<7' 'PyYAML>=6,<7' 'joblib>=1.3,<2'

## Make the project code available
Upload this project folder to Colab or mount your private Google Drive. Mounting is optional and sends data to Google infrastructure, so do it only if privacy and data-use permissions allow it.

In [ ]:
from pathlib import Path
PROJECT_DIR = Path('/content/osteoinsight')  # change to the uploaded project folder
DATASET_DIR = Path('/content/authorized_dataset')  # change to the explicitly authorized dataset
assert (PROJECT_DIR / 'audit_roboflow_classification.py').exists(), 'Upload/copy the project first'
assert DATASET_DIR.is_dir(), 'Set DATASET_DIR to the authorized image folder'
%cd $PROJECT_DIR

## Read-only audit
The reports folder must be outside the dataset. Folder names remain unverified metadata in this first pass.

In [ ]:
!python audit_roboflow_classification.py "$DATASET_DIR" --output-dir reports/roboflow_audit --expected-classes normal cancer
print((Path('reports/roboflow_audit') / 'dataset_report.md').read_text())

## Scenario C — no reliable labels (safe default)
Run pretrained feature extraction and visual clustering. Clusters are not diagnoses. Adjust the cluster count for review, not medical interpretation.

In [ ]:
!python explore_embeddings.py reports/dataset_manifest.csv --output-dir reports/embeddings --clusters 8 --batch-size 64

## Scenario A — verified classification labels only
Run these cells only after documenting label provenance and confirming the categories are valid ground truth. Re-audit with the explicit trust flag, make leakage-aware splits, train, and evaluate.

In [ ]:
import json
audit = json.loads(Path('reports/roboflow_audit/audit_summary.json').read_text())
assert audit['valid_one_hot_rows'] == audit['rows'], 'Resolve invalid CSV labels before training'
assert audit['training_gate_pass'], audit['training_gate_reason']
print('Training dataset categories only; labels are not independently verified clinical diagnoses.')
!python train.py --roboflow-root "$DATASET_DIR" --architecture resnet18 --output-dir models --epochs 30 --batch-size 64
!python evaluate.py --roboflow-root "$DATASET_DIR" --checkpoint models/osteoinsight_best.pth --output-dir reports/evaluation --positive-class cancer

## Application
Use the local app whenever possible for private data. A public share URL is intentionally not enabled.

**Research prototype only. This system is not validated for clinical diagnosis, treatment decisions, or patient prognosis. Results require professional medical interpretation.**

In [ ]:
!python app.py